# 검수한 데이터를 학습 · 평가 · 시험으로 나누기
준비: `pairs_checked.csv` — 열: 본문, 리드, 판정(사용·수정·제외)
- **수정**한 쌍은 `리드` 칸을 고친 문장으로 바꿔 두세요.

In [6]:
# ▶ 환경 준비 — Colab · VS Code 공통, 맨 처음 한 번 실행
import os, sys, platform, subprocess
IN_COLAB = 'google.colab' in sys.modules

def gpu_info():
    try:
        r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                           capture_output=True, text=True, timeout=10)
        return r.stdout.strip() or None
    except Exception:
        return None

GPU = gpu_info()
print('실행 위치:', 'Colab' if IN_COLAB else f'내 컴퓨터 ({platform.system()})')
print('GPU:', GPU or '없음 → CPU로 실행 (이 노트북은 CPU로도 충분해요)')

# (선택) pandas를 GPU로 — Colab GPU 런타임 또는 Linux/WSL에 cuDF가 있을 때만
USE_GPU_PANDAS = False          # True로 바꾸고 이 칸을 다시 실행
if USE_GPU_PANDAS and GPU:
    try:
        get_ipython().run_line_magic('load_ext', 'cudf.pandas')
        print('pandas → GPU(cuDF) 사용')
    except Exception as e:
        print('cuDF를 쓸 수 없어 일반 pandas로 실행:', type(e).__name__)

# 한글 글꼴: Colab은 나눔고딕 설치, 윈도우는 맑은 고딕, 맥은 애플고딕
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm
if IN_COLAB:
    nanum = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
    if not os.path.exists(nanum):
        subprocess.run('apt-get -qq install -y fonts-nanum > /dev/null', shell=True)
    fm.fontManager.addfont(nanum)
    plt.rcParams['font.family'] = 'NanumGothic'
elif platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:                                   # 리눅스·WSL: 설치된 한글 글꼴 찾기
    ko = [f.name for f in fm.fontManager.ttflist if 'Nanum' in f.name or 'CJK KR' in f.name]
    if ko: plt.rcParams['font.family'] = ko[0]
plt.rcParams['axes.unicode_minus'] = False
print('한글 글꼴:', plt.rcParams['font.family'])

실행 위치: 내 컴퓨터 (Windows)
GPU: NVIDIA GeForce RTX 5070 Laptop GPU, 8151 MiB
한글 글꼴: ['Malgun Gothic']


In [7]:
import pandas as pd
df = pd.read_csv('pairs_checked.csv')
print(df['판정'].value_counts())

판정
사용    313
제외     41
Name: count, dtype: int64


In [ ]:
df = df[df['판정'] != '제외'].copy()                           # 제외한 쌍 빼기
# df['instruction'] = '보도자료를 읽고 기사 첫 문장(리드)을 한 문장으로 쓰세요.'
df['instruction'] = '스포츠 리포트 본문을 읽고, 앵커가 읽을 도입 멘트(리드)를 MBC 뉴스 말투로 쓰세요.'
df['input'], df['output'] = df['본문'], df['리드']
test  = df.sample(5, random_state=42)                      # 마지막 비교용, 학습에 절대 안 씀
rest  = df.drop(test.index)
eval_ = rest.sample(frac=0.15, random_state=42)
train = rest.drop(eval_.index)
for d, name in [(train, 'train.jsonl'), (eval_, 'eval.jsonl'), (test, 'test.jsonl')]:
    d[['instruction', 'input', 'output']].to_json(name, orient='records', lines=True, force_ascii=False)
print(len(train), len(eval_), len(test))

262 46 5


## (VS Code · 강의장 노트북) 세 모델 답 자동으로 받기
Ollama는 GPU가 있으면 알아서 GPU로 실행해요. 먼저 터미널에서 `ollama list`로 모델 이름을 확인하세요.
- 원본: `ollama pull qwen2.5:3b` · 내 모델: `ollama create 내모델이름 -f Modelfile`

In [3]:
# ▶ API 키 불러오기 — Colab은 🔑 보안 비밀, VS Code는 같은 폴더의 .env 파일
#   .env 파일 내용 예:  OPENAI_API_KEY=sk-...   (GitHub에 올리지 않기)
def get_key(name='OPENAI_API_KEY'):
    if IN_COLAB:
        from google.colab import userdata
        return userdata.get(name)
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        print('pip install python-dotenv 후 다시 실행')
    return os.getenv(name)

OPENAI_API_KEY = get_key()
print('API 키:', '준비됨' if OPENAI_API_KEY else '없음 → 키를 등록하세요')

API 키: 준비됨


In [4]:
from openai import OpenAI
import pandas as pd
test = pd.read_json('test.jsonl', lines=True)
local = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')   # 내 컴퓨터의 Ollama
cloud = OpenAI(api_key=OPENAI_API_KEY) if OPENAI_API_KEY else None

def ask(client, model, row):
    msg = row['instruction'] + '\n\n' + row['input']
    try:
        r = client.chat.completions.create(model=model, messages=[{'role': 'user', 'content': msg}], temperature=0)
        return r.choices[0].message.content.strip()
    except Exception as e:
        return f'(실패: {type(e).__name__})'

MODELS = {'원본 Qwen2.5-3B': (local, 'qwen2.5:3b'),
          '내가 학습한 모델': (local, '내모델이름'),          # ollama list 에 나온 이름으로 바꾸기
          'gpt-4.1-mini':    (cloud, 'gpt-4.1-mini')}
answers = {name: [ask(c, m, r) if c else '(키 없음)' for _, r in test.iterrows()]
           for name, (c, m) in MODELS.items()}
pd.DataFrame(answers).head()

FileNotFoundError: [Errno 2] No such file or directory: 'test.jsonl'

## 블라인드 비교표 만들기 — 세 모델 답을 섞어서
위 칸을 실행했다면 `answers`가 이미 채워져 있어요. 아니면 아래에 직접 붙여 넣으세요.

In [5]:
import random
# 위 칸을 안 썼다면: 시험 5문제에 대한 세 모델의 답을 직접 붙여 넣으세요
if 'answers' not in globals(): answers = {
    '원본 Qwen2.5-3B': ['', '', '', '', ''],
    '내가 학습한 모델': ['', '', '', '', ''],
    'gpt-4.1-mini':    ['', '', '', '', ''],
}
rows, key = [], []
for i in range(5):
    names = list(answers); random.Random(i).shuffle(names)
    rows.append({'문제': i + 1, **{k: answers[n][i] for k, n in zip('ABC', names)}})
    key.append({'문제': i + 1, **dict(zip('ABC', names))})
pd.DataFrame(rows).to_csv('blind_compare.csv', index=False, encoding='utf-8-sig')
pd.DataFrame(key).to_csv('blind_answer_key.csv', index=False, encoding='utf-8-sig')   # 다 고른 뒤 열어 보기